In [ ]:
import os, sys
from urllib.parse import urlparse
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType
from shapely.wkt import loads
from shapely.geometry import Point

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [ ]:
path_munic_coord_geo = r"C:\Marco Conti\Projetos\Dados\municipios\municipios_coord_geograficas.parquet"
path_munic_bioma     = r"C:\Marco Conti\Projetos\Dados\municipios\tb_munic_sit_bioma.parquet"

df_munic_coord_geo = spark.read.parquet(path_munic_coord_geo)
df_munic_coord_geo.printSchema()
df_munic_coord_geo.show(10, truncate=False)

path_ondas_calor = r"C:\Marco Conti\Projetos\Dados\ondas_calor\Ondas_Calor_Anual.parquet"

df_ondas_calor = spark.read.parquet(path_ondas_calor)
df_ondas_calor.printSchema()
df_ondas_calor.show(3, False)


# df_munic_brasil_lat_long = \
#     spark.read.csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\municipios_brasil_lat_long.csv"
#                   ,header=True
#                   ,inferSchema=True)

# df_temp_mensal = \
#     spark.read.csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\tb_temperatura_mensal.csv"
#                   ,header=True
#                   ,inferSchema=True
#                   ,sep=";")

In [ ]:
df_munic_coord_geo.printSchema()
df_ondas_calor.printSchema()

In [ ]:
df_ondas_calor.filter("latitude = -23 and longitude = -46").show()
df_munic_coord_geo.filter("nome_municipio like '%Tab%Serra%'").show()

In [ ]:
# Arredondar as colunas de latitude e longitude para 4 casas decimais, ficará mais próximo dos dados do ERA5, que é de 25 em 25 graus
df_munic_brasil_lat_long = \
    (df_munic_coord_geo
        .withColumn("latitude", F.round(df_munic_coord_geo["latitude"], 2))
        .withColumn("longitude", F.round(df_munic_coord_geo["longitude"], 2)))

df_munic_brasil_lat_long_grades = \
    (df_munic_brasil_lat_long
        .withColumn("lat_grade", F.round(F.col("latitude") * 4) / 4)
        .withColumn("lon_grade", F.round(F.col("longitude") * 4) / 4)
    )

df_munic_brasil_lat_long_grades.printSchema()
df_munic_brasil_lat_long_grades.filter("codigo_municipio = 3552809").limit(10).show(truncate=False)

# df_temp_mensal.printSchema()
# df_temp_mensal.limit(10).show(truncate=False)

In [ ]:
# print("Número de registros do DataFrame de municípios: ", df_munic_brasil_lat_long_grades.count())
# print("Número de registros do DataFrame de temperatura mensal: ", df_temp_mensal.count())

df_munic_brasil_lat_long_grades.printSchema()
df_ondas_calor.printSchema()

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_ibge = df_munic_brasil_lat_long
df_era5 = df_ondas_calor


# --- CONFIGURAÇÃO ---
RAIO_BUSCA_GRAUS = 0.4 

# 1. RENOMEAR COLUNAS DO IBGE PARA EVITAR DUPLICIDADE
df_ibge_renomeado = \
    (df_ibge
        .withColumnRenamed("latitude" , "lat_ibge")
        .withColumnRenamed("longitude", "lon_ibge")
        .withColumnRenamed("ano"      , "ano_ibge")
    )

# 2. EXECUTAR O JOIN COM AS COLUNAS JÁ DISTINTAS
df_cruzado = \
    (df_era5.join(F.broadcast(df_ibge_renomeado)
        ,on=[df_era5["ano"] == df_ibge_renomeado["ano_ibge"]
           ,df_era5["latitude"].between(df_ibge_renomeado["lat_ibge"] - RAIO_BUSCA_GRAUS, df_ibge_renomeado["lat_ibge"] + RAIO_BUSCA_GRAUS)
           ,df_era5["longitude"].between(df_ibge_renomeado["lon_ibge"] - RAIO_BUSCA_GRAUS, df_ibge_renomeado["lon_ibge"] + RAIO_BUSCA_GRAUS)
           ]
        ,how="inner")
    )

# 3. CÁLCULO DA DISTÂNCIA REAL (Haversine em KM)
# Agora as referências são diretas e seguras
lat1 = F.radians(F.col("latitude"))      # Vem do ERA5
lon1 = F.radians(F.col("longitude"))     # Vem do ERA5
lat2 = F.radians(F.col("lat_ibge"))      # Vem do IBGE
lon2 = F.radians(F.col("lon_ibge"))      # Vem do IBGE

dlat = lat2 - lat1
dlon = lon2 - lon1

# Fórmula de Haversine
a = F.sin(dlat / 2)**2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2)**2
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
R = 6371.0  # Raio da Terra em km

df_com_distancia = df_cruzado.withColumn("distancia_km", c * R)

# 4. SELECIONAR APENAS O VIZINHO MAIS PRÓXIMO
window_spec = \
    (Window.partitionBy("codigo_municipio"
                       ,"ano")
            .orderBy("distancia_km"))

df_resultado_final = \
    (df_com_distancia
        .withColumn("rank", F.row_number().over(window_spec))
        .filter(F.col("rank") == 1)
        .drop("rank"))


In [ ]:
df_resultado_final.filter("codigo_municipio = 3552809").show()

In [ ]:
-23.0   |-46.0    |2024|2                 |7                         |4                  |3.57               |25.417169189453148  |24.85               |

In [ ]:
# Exibir o resultado limpo
(df_resultado_final
    .filter("codigo_municipio like '110012%'")
    .select("codigo_municipio"
           ,"nome_municipio"
           ,"uf"
           ,"ano"
           ,"mes"
           ,"temp_min_mes"
           ,"temp_max_mes"
           ,"temp_media_mes"
           ,"percentil_05_mes"
           ,"percentil_90_mes"
           ,F.col("latitude").alias("era5_lat")
           ,F.col("longitude").alias("era5_lon")
           ,F.col("lat_ibge").alias("ibge_lat")
           ,F.col("lon_ibge").alias("ibge_lon")
           ,F.col("temp_media_mes")
           ,F.col("distancia_km"))
        .show())

In [ ]:
df_resultado_final.filter("temp_media_mes is not null").count() 

# 11140 Total de registros 
#  5570 Municípios 